# 13. Transforms

*Adapted from the official [Transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html) tutorial. This topic isn't in Raschka's primer at all - our toy tensors in notebooks 01-09 never needed preprocessing - but it's essential once you work with real datasets like FashionMNIST (notebooks 10, 12).*

Raw data rarely comes in the final form training needs. **Transforms** perform that preprocessing.
Every `torchvision` dataset takes two callables: `transform` (for the features/images) and
`target_transform` (for the labels).

```mermaid
flowchart LR
    A["Raw PIL image<br/>(0-255 uint8)"] --> B["v2.ToImage()<br/>-> tv_tensors.Image"]
    B --> C["v2.ToDtype(float32,<br/>scale=True)<br/>-> [0.0, 1.0]"]
    C --> D["Augmentation<br/>(Flip/Resize/etc)"]
    D --> E["v2.Normalize<br/>(mean/std)"]
    E --> F["Tensor ready<br/>for the model"]
    F --> G["Batched by<br/>DataLoader<br/>(notebook 12)"]
```

## `ToImage()` and `ToDtype()`

FashionMNIST features arrive as PIL images; labels arrive as plain integers. For training we want
float tensors (normalized to [0, 1]) and, if we want one-hot labels, a small `Lambda` transform.

The modern `torchvision.transforms.v2` API splits what used to be a single `ToTensor()` call into
two explicit steps:

- **`v2.ToImage()`** converts a PIL image / NumPy array into a `torchvision.tv_tensors.Image`.
- **`v2.ToDtype(torch.float32, scale=True)`** casts to `float32` and rescales pixel values from
  `[0, 255]` to `[0.0, 1.0]`.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import v2

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    target_transform=v2.Lambda(
        lambda y: F.one_hot(torch.tensor(y), num_classes=10).float()
    ),
)

image, one_hot_label = ds[0]
print("image shape/dtype:", image.shape, image.dtype, "range:", image.min().item(), "-", image.max().item())
print("one-hot label:", one_hot_label)


## Lambda transforms

`v2.Lambda` wraps any user-defined function as a transform. Above we used
`torch.nn.functional.one_hot` to turn an integer class label (0-9) into a 10-element one-hot float
vector - matching the shape a loss function like `nn.MSELoss` would expect, as opposed to
`nn.CrossEntropyLoss`, which expects raw integer class indices (which is why notebooks 06/10/16 in
this primer *don't* one-hot-encode their labels - `cross_entropy` wants the plain integer form).

## Chaining geometric/augmentation transforms

In practice you'll almost always chain several transforms beyond just `ToImage`/`ToDtype` —
resizing, flipping for data augmentation, and normalizing. Let's build a fuller pipeline and
actually look at what it does to an image, before vs. after:

In [ ]:
import matplotlib.pyplot as plt

raw_ds = datasets.FashionMNIST(root="data", train=True, download=True)  # no transform: raw PIL
raw_image, _ = raw_ds[0]

augment_pipeline = v2.Compose([
    v2.Resize((32, 32)),                          # e.g. to match a model's expected input size
    v2.RandomHorizontalFlip(p=1.0),                # p=1.0 here only so the demo always flips
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5], std=[0.5]),           # rescales [0,1] -> roughly [-1, 1]
])
augmented = augment_pipeline(raw_image)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(raw_image, cmap="gray")
axes[0].set_title(f"Before: {raw_image.size}")
axes[0].axis("off")
# unnormalize just for display: [-1,1] -> [0,1]
axes[1].imshow((augmented.squeeze() * 0.5 + 0.5), cmap="gray")
axes[1].set_title(f"After: {tuple(augmented.shape)}")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## A custom transform as a callable class, vs. `v2.Lambda`

`v2.Lambda` is great for a one-off function, but for a transform with configuration (e.g. a
specific mean/std) a plain callable class is the more standard, reusable pattern in real projects:

In [ ]:
class NormalizeByStats:
    """Callable-class transform: reusable, configurable, no lambda needed."""
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return (tensor - self.mean) / self.std


# Equivalent to v2.Normalize(mean=[0.5], std=[0.5]) above, but as a class we control ourselves:
custom_normalize = NormalizeByStats(mean=0.5, std=0.5)
sample_tensor = v2.ToDtype(torch.float32, scale=True)(v2.ToImage()(raw_image))
print("custom class output range:", custom_normalize(sample_tensor).min().item(),
      "to", custom_normalize(sample_tensor).max().item())


## Gotcha: `v2.Compose` order matters

Transforms run in the exact order you list them — swapping two steps can silently change the
result. A classic mistake: normalizing *before* scaling to `[0, 1]` instead of after:

In [ ]:
# Correct order: convert to image tensor, scale to [0,1], THEN normalize.
correct_pipeline = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),   # [0, 255] uint8 -> [0.0, 1.0] float
    v2.Normalize(mean=[0.5], std=[0.5]),      # [0.0, 1.0] -> roughly [-1, 1]
])
correct_out = correct_pipeline(raw_image)
print(f"Correct order -> range [{correct_out.min():.2f}, {correct_out.max():.2f}]  (expected ~[-1, 1])")

# WRONG order: Normalize BEFORE the dtype/scale conversion. Normalize expects a float
# tensor already in a known range — feeding it the raw uint8 image tensor either errors
# outright or silently normalizes around the wrong scale, depending on the torchvision version.
wrong_pipeline = v2.Compose([
    v2.ToImage(),
    v2.Normalize(mean=[0.5], std=[0.5]),      # too early: still uint8 [0, 255] here
    v2.ToDtype(torch.float32, scale=True),
])
try:
    wrong_out = wrong_pipeline(raw_image)
    print(f"Wrong order   -> range [{wrong_out.min():.2f}, {wrong_out.max():.2f}]  (NOT ~[-1, 1] — order bug)")
except Exception as e:
    print(f"Wrong order raised {type(e).__name__}: {e}")


## Further reading

- [Getting started with transforms v2](https://pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_getting_started.html)
- [torchvision.transforms.v2 API reference](https://pytorch.org/vision/stable/transforms.html#v2-api-reference-recommended)